# Code_3 — Schema-Only Detection and Correction: Core Experiments

**GPU: no** (CPU runtime; the MiniLM function embeddings run fine on CPU)

Everything here uses **only the schema registry (and, where stated, the user query)** at
inference time. Ground truth is used only after the pipeline has produced its output, to score it.

| Section | Mentor point |
|---|---|
| A. Tuning on the tune part only (type-trust threshold, matcher weights, number of passes) | #5, #11 |
| B. Main results on both test splits: detection, classification, correction, intent proxy | #1, #3, #4 |
| C. Format check before vs after normalisation | #12 |
| D. Ablation (each component switched off, with McNemar tests) | #5 |
| E. Function-matcher weight sensitivity | #5 |
| F. Type-checking strategies and the adaptive threshold sweep | #11 |
| G. Schema-quality robustness (0–30 % corrupted declarations) | #10 |
| H. Multi-error calls (k = 1…4) | #2 |
| I. WRONG_VALUE: schema only vs schema + query | #6 |
| J. Failure categories | — |
| K. Runtime per component | #15 |

Output: `runs/<run_id>/core_results.json` and `runs/<run_id>/core_records.json` (per-example outcomes).

In [ ]:
import os, sys, json, importlib, subprocess, time, copy, random
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'rapidfuzz'], check=True)
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)
import numpy as np

NOTEBOOK = 'Code_3'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
th.set_seed(SEED)
NB, ALPHA = CFG['bootstrap']['n'], CFG['bootstrap']['alpha']

raw_registry = th.load_json(P.data_file('registry.json'), CFG, kind='data')
single = th.load_json(P.data_file('benchmark_single.json'), CFG, kind='data')
multi = th.load_json(P.data_file('benchmark_multi.json'), CFG, kind='data')
splits = th.load_json(P.data_file('splits.json'), CFG, kind='data')
REG = th.Registry(raw_registry, CFG['validator']['unknown_required_policy'])
print(f"run_id={CFG['run_id']} | registry {len(REG)} functions | single {len(single)} | multi {len(multi)}")

t0 = time.time()
BASE_MATCHER = th.FunctionMatcher(REG, model_name=CFG['matcher']['model_name'], cache_dir=P.cache,
                                  use_semantic=CFG['matcher']['use_semantic'],
                                  use_edit=CFG['matcher']['use_edit'],
                                  w_semantic=CFG['matcher']['w_semantic'],
                                  min_score=CFG['matcher']['min_score'], topk=CFG['matcher']['topk'])
print(f"matcher ready in {time.time() - t0:.0f}s | semantic={BASE_MATCHER.use_semantic}")
SEMANTIC_AVAILABLE = BASE_MATCHER.use_semantic
RESULTS = {'hardware': th.hardware_info(), 'packages': th.package_versions(),
           'semantic_matching_available': SEMANTIC_AVAILABLE}

## Helpers

In [ ]:
def subset(idx, data=single):
    return [data[i] for i in idx]


def fit_trust(tune_examples, mode, tau=None, registry=REG):
    tcfg = CFG['type_trust']
    tt = th.TypeTrust(mode=mode, tau=tcfg['tau'] if tau is None else tau, min_obs=tcfg['min_obs'],
                      prior_agree=tcfg['prior_agree'], prior_disagree=tcfg['prior_disagree'])
    clean_calls = [th.normalize_tool_call(ex['generated_tool_call'])
                   for ex in tune_examples if not ex['is_hallucinated']]
    return tt.fit(clean_calls, registry)


def make_pipe(trust, matcher=None, vo=None, co=None, detect_only=False, registry=REG):
    p = th.Pipeline(registry, CFG, type_trust=trust, matcher=matcher if matcher is not None else BASE_MATCHER,
                    validator_overrides=vo, corrector_overrides=co)
    p.detect_only = detect_only
    return p


def summarize(records):
    b = th.binary_metrics([r['is_hallucinated'] for r in records], [r['flagged'] for r in records])
    hall = [r for r in records if r['is_hallucinated']]
    clean = [r for r in records if not r['is_hallucinated']]
    single_lab = [r for r in records if len(r['true_types']) <= 1]
    mc = th.multiclass_metrics([r['true_types'][0] if r['true_types'] else 'NONE' for r in single_lab],
                               [r['pred_primary'] for r in single_lab]) if single_lab else {'macro_f1': None}
    mean = lambda rs, k: float(np.mean([bool(r[k]) for r in rs])) if rs else None
    return {'n': len(records), 'det_accuracy': b['accuracy'], 'det_balanced_accuracy': b['balanced_accuracy'],
            'det_precision': b['precision'], 'det_recall': b['recall'], 'det_f1': b['f1'],
            'fpr': b['fpr'], 'fnr': b['fnr'], 'class_macro_f1': mc['macro_f1'],
            'exact_all': mean(records, 'final_exact'), 'exact_hallucinated': mean(hall, 'final_exact'),
            'exact_clean': mean(clean, 'final_exact'),
            'schema_valid_hallucinated': mean(hall, 'final_schema_valid')}


def compare(records_a, records_b):
    """McNemar + paired bootstrap on final_exact, a vs b (same examples, same order)."""
    assert [r['id'] for r in records_a] == [r['id'] for r in records_b]
    a = [int(r['final_exact']) for r in records_a]
    b = [int(r['final_exact']) for r in records_b]
    return {'mcnemar': th.mcnemar_exact(a, b),
            'diff_ci': th.paired_bootstrap_diff(a, b, n=NB, alpha=ALPHA, seed=SEED)}


def print_row(name, s, extra=''):
    f = lambda v: f"{v:7.3f}" if isinstance(v, float) else f"{'—':>7}"
    print(f"{name:<30}{f(s['det_f1'])}{f(s['det_balanced_accuracy'])}{f(s['fpr'])}{f(s['fnr'])}"
          f"{f(s['class_macro_f1'])}{f(s['exact_all'])}{f(s['exact_hallucinated'])}"
          f"{f(s['schema_valid_hallucinated'])}  {extra}")


HEAD = (f"{'variant':<30}{'detF1':>7}{'balAcc':>7}{'FPR':>7}{'FNR':>7}{'macF1':>7}"
        f"{'exact':>7}{'exH':>7}{'svH':>7}")

## A. Tuning — on the tune part only

Three choices are made on the tune part of each split, in this order, and then frozen:
1. **type-trust threshold τ** (adaptive type checking) — maximises detection balanced accuracy;
2. **function-matcher weights** (semantic weight, minimum score) — maximise end-to-end exact rate;
3. **maximum correction passes** — maximises end-to-end exact rate.

In [ ]:
TUNED = {}
TUNING_CURVES = {}
for split_name in ['random', 'function_disjoint']:
    tune = subset(splits[split_name]['tune'])
    curves = {}
    best = None
    curves['tau'] = []
    for tau in CFG['type_trust']['tau_grid']:
        tt = fit_trust(tune, 'adaptive', tau)
        s = summarize(make_pipe(tt, detect_only=True).run(tune))
        curves['tau'].append({'tau': tau, **s})
        if best is None or s['det_balanced_accuracy'] > best[1] + 1e-12:
            best = (tau, s['det_balanced_accuracy'])
    tau_sel = best[0]
    trust = fit_trust(tune, 'adaptive', tau_sel)
    curves['matcher'] = []
    w_grid = CFG['matcher']['w_semantic_grid'] if SEMANTIC_AVAILABLE else [0.0]
    best = None
    for w in w_grid:
        for ms in CFG['matcher']['min_score_grid']:
            m = BASE_MATCHER.configured(w_sem=w, min_score=ms, use_semantic=SEMANTIC_AVAILABLE and w > 0,
                                        use_edit=w < 1.0)
            s = summarize(make_pipe(trust, m).run(tune))
            curves['matcher'].append({'w_semantic': w, 'min_score': ms, **s})
            if best is None or s['exact_all'] > best[2] + 1e-12:
                best = (w, ms, s['exact_all'])
    w_sel, ms_sel = best[0], best[1]
    matcher = BASE_MATCHER.configured(w_sem=w_sel, min_score=ms_sel,
                                      use_semantic=SEMANTIC_AVAILABLE and w_sel > 0, use_edit=w_sel < 1.0)
    curves['max_iterations'] = []
    best = None
    for it in CFG['corrector']['max_iterations_grid']:
        s = summarize(make_pipe(trust, matcher, co={'max_iterations': it}).run(tune))
        curves['max_iterations'].append({'max_iterations': it, **s})
        if best is None or s['exact_all'] > best[1] + 1e-12:
            best = (it, s['exact_all'])
    it_sel = best[0]
    TUNED[split_name] = {'tau': tau_sel, 'w_semantic': w_sel, 'min_score': ms_sel, 'max_iterations': it_sel,
                         'trust_summary': trust.summary()}
    TUNING_CURVES[split_name] = curves
    print(f"[{split_name}] tuned on {len(tune)} tune examples -> tau={tau_sel}, w_semantic={w_sel}, "
          f"min_score={ms_sel}, max_iterations={it_sel}")
    print(f"   type trust: {trust.summary()['pairs_contradicted']} of {trust.summary()['pairs_seen']} "
          f"(function, param) declarations contradicted by clean tune calls")
RESULTS['tuned'] = TUNED
RESULTS['tuning_curves'] = TUNING_CURVES


def tuned_parts(split_name):
    t = TUNED[split_name]
    tune = subset(splits[split_name]['tune'])
    trust = fit_trust(tune, 'adaptive', t['tau'])
    matcher = BASE_MATCHER.configured(w_sem=t['w_semantic'], min_score=t['min_score'],
                                      use_semantic=SEMANTIC_AVAILABLE and t['w_semantic'] > 0,
                                      use_edit=t['w_semantic'] < 1.0)
    return trust, matcher, {'max_iterations': t['max_iterations']}

## B. Main results on the test parts (frozen configuration)

Four separate levels (#4):
1. **Detection** — is the call wrong? (binary, with 95 % bootstrap CIs)
2. **Classification** — which error type? (7-class confusion matrix, macro-F1)
3. **Correction** — is the output schema-valid?
4. **Intent proxy** — does the output equal the reference call? (exact match, envelope included).
   A schema-valid output is *not* necessarily correct; the gap between levels 3 and 4 is reported.

In [ ]:
MAIN, MAIN_RECORDS = {}, {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    trust, matcher, co = tuned_parts(split_name)
    timers = th.new_timers()
    recs = make_pipe(trust, matcher.fresh(), co=co).run(test, timers)
    MAIN_RECORDS[split_name] = recs
    det = th.detection_report(recs, NB, ALPHA, SEED)
    cor = th.correction_report(recs, NB, ALPHA, SEED)
    MAIN[split_name] = {'detection': det, 'correction': cor, 'latency': th.latency_summary(timers)}
    b = det['binary']
    th.print_header(f"MAIN RESULT — {split_name} test split (n={len(test)})")
    print("Level 1  detection")
    print(f"   accuracy {th.fmt_ci(det['ci']['accuracy'])} | balanced acc {th.fmt_ci(det['ci']['balanced_accuracy'])}"
          f" | F1 {th.fmt_ci(det['ci']['f1'])}")
    print(f"   precision {b['precision']:.3f} recall {b['recall']:.3f} FPR {b['fpr']:.3f} FNR {b['fnr']:.3f}")
    mc = det['classification']
    print(f"Level 2  classification   macro-F1 {th.fmt_ci(mc['ci_macro_f1'])} | 7-class acc {mc['accuracy']:.3f}"
          f" | balanced acc {mc['balanced_accuracy']:.3f}")
    print(f"   {'class':<18}{'prec':>7}{'rec':>7}{'F1':>7}{'n':>6}")
    for lab in th.ALL_LABELS:
        pc = mc['per_class'][lab]
        print(f"   {lab:<18}{pc['precision']:>7.3f}{pc['recall']:>7.3f}{pc['f1']:>7.3f}{pc['support']:>6}")
    print("   confusion matrix (rows = true, columns = predicted):")
    print("   " + " " * 18 + ''.join(f"{l[:6]:>8}" for l in mc['labels']))
    for lab, row in zip(mc['labels'], mc['confusion_matrix']):
        print(f"   {lab:<18}" + ''.join(f"{v:>8}" for v in row))
    print(f"Level 3  correction       schema-valid output (hallucinated calls): "
          f"{cor['final_schema_valid_rate_hallucinated']:.3f}")
    print(f"Level 4  intent proxy     exact match, all calls: {th.fmt_ci(cor['ci']['final_exact_rate_all'])}"
          f" | hallucinated: {th.fmt_ci(cor['ci']['final_exact_rate_hallucinated'])}"
          f" | clean kept correct: {cor['final_exact_rate_clean']:.3f}")
    print(f"   schema-valid but NOT exact (hallucinated): {cor['schema_valid_but_not_exact_hallucinated']:.3f}")
    print(f"\n   {'true type':<18}{'n':>5}{'flagged':>9}{'typeOK':>8}{'schemaValid':>12}{'exact':>7}")
    for lab in th.ALL_LABELS:
        d = det['by_true_type'].get(lab)
        c = cor['by_true_type'].get(lab)
        if d:
            print(f"   {lab:<18}{d['n']:>5}{d['flag_rate']:>9.3f}{d['type_correct_rate']:>8.3f}"
                  f"{c['final_schema_valid']:>12.3f}{c['final_exact']:>7.3f}")
RESULTS['main'] = MAIN

## C. Format check before vs after normalisation (#12)

If the envelope is checked **after** normalisation, every wrong envelope has already been rewritten
into `{function, parameters}` and the error disappears. The pipeline therefore checks the raw
envelope first: raw call → parse → **format check** → normalise → schema checks → classify → correct.

In [ ]:
fmt = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    wf = [ex for ex in test if ex['error_types'] == ['WRONG_FORMAT']]
    v = th.Validator(REG, CFG['validator'], CFG['envelope']['accepted_key_sets'])
    raw_first = [any(x['type'] == 'WRONG_FORMAT' for x in v.validate(ex['generated_tool_call'])) for ex in wf]
    norm_first = [any(x['type'] == 'WRONG_FORMAT' for x in v.validate(th.normalize_tool_call(ex['generated_tool_call'])))
                  for ex in wf]
    by_env = defaultdict(lambda: [0, 0])
    for ex, d in zip(wf, raw_first):
        e = ex['injection'][0]['envelope']
        by_env[e][0] += int(d)
        by_env[e][1] += 1
    fmt[split_name] = {'n_wrong_format': len(wf), 'detected_format_first': float(np.mean(raw_first)) if wf else None,
                       'detected_normalise_first': float(np.mean(norm_first)) if wf else None,
                       'by_envelope': {k: {'detected': a, 'n': n} for k, (a, n) in by_env.items()}}
    print(f"[{split_name}] WRONG_FORMAT detected: format-check-first {fmt[split_name]['detected_format_first']:.3f}"
          f" vs normalise-first {fmt[split_name]['detected_normalise_first']:.3f}  by envelope {dict(fmt[split_name]['by_envelope'])}")
RESULTS['format_order'] = fmt

## D. Ablation (#5)

Every variant uses the frozen tuned configuration with **one** change. Δ = variant − full on
end-to-end exact match, with an exact McNemar test and a paired-bootstrap 95 % CI.
A positive Δ means removing/changing that component *helped*.

In [ ]:
def ablation_variants(trust_tuned, matcher_tuned, co_tuned, tune_examples):
    V = {'full': dict()}
    V['detect_only (no correction)'] = dict(detect_only=True)
    V['single correction pass'] = dict(co={'max_iterations': 1})
    for it in CFG['corrector']['max_iterations_grid']:
        V[f'max passes = {it}'] = dict(co={'max_iterations': it})
    V['fix all violations per pass'] = dict(co={'fixes_per_iteration': 'all'})
    V['no type check'] = dict(vo={'check_type': False})
    V['type check strict (no calibration)'] = dict(trust=fit_trust(tune_examples, 'strict'))
    V['type check skip-list (old method)'] = dict(trust=fit_trust(tune_examples, 'skiplist'))
    V['no format check'] = dict(vo={'check_format': False})
    V['no format fix'] = dict(co={'fix_format': False})
    V['no function correction'] = dict(co={'fix_function': False})
    V['no extra-param removal'] = dict(co={'fix_extra': False})
    V['no missing-param fill'] = dict(co={'fix_missing': False})
    V['no type cast'] = dict(co={'fix_type': False})
    V['matcher: edit distance only'] = dict(matcher=matcher_tuned.configured(use_semantic=False, use_edit=True))
    if SEMANTIC_AVAILABLE:
        V['matcher: semantic only'] = dict(matcher=matcher_tuned.configured(use_semantic=True, use_edit=False))
    V['+ query grounding (schema+query)'] = dict(vo={'check_grounding': True}, co={'fix_value_from_query': True})
    return V


ABL = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    tune = subset(splits[split_name]['tune'])
    trust, matcher, co = tuned_parts(split_name)
    full_recs = MAIN_RECORDS[split_name]
    th.print_header(f"ABLATION — {split_name} test split")
    print(HEAD + "   Δexact [95% CI]   McNemar p")
    ABL[split_name] = {}
    for name, spec in ablation_variants(trust, matcher, co, tune).items():
        co_v = {**co, **spec.get('co', {})}
        recs = full_recs if name == 'full' else make_pipe(spec.get('trust', trust), spec.get('matcher', matcher),
                                                          vo=spec.get('vo'), co=co_v,
                                                          detect_only=spec.get('detect_only', False)).run(test)
        s = summarize(recs)
        cmpr = compare(recs, full_recs) if name != 'full' else None
        ABL[split_name][name] = {'summary': s, 'vs_full': cmpr}
        extra = '' if cmpr is None else (f"{cmpr['diff_ci']['diff']:+.3f} [{cmpr['diff_ci']['low']:+.3f},"
                                          f"{cmpr['diff_ci']['high']:+.3f}]  p={cmpr['mcnemar']['p_value']:.2g}")
        print_row(name, s, extra)
RESULTS['ablation'] = ABL

## E. Function-matcher weight sensitivity (#5)

`score = w · cosine(embedding) + (1 − w) · (1 − normalised edit distance)`, accepted if ≥ min_score.
The grid was evaluated on the **tune** part in section A (selection); the same grid is shown on the
**test** part here for transparency only — test numbers were not used to choose anything.

In [ ]:
SENS = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    trust, _, co = tuned_parts(split_name)
    wf_test = [ex for ex in test if 'WRONG_FUNCTION' in ex['error_types']]
    rows = []
    w_grid = CFG['matcher']['w_semantic_grid'] if SEMANTIC_AVAILABLE else [0.0]
    for w in w_grid:
        for ms in CFG['matcher']['min_score_grid']:
            m = BASE_MATCHER.configured(w_sem=w, min_score=ms, use_semantic=SEMANTIC_AVAILABLE and w > 0,
                                        use_edit=w < 1.0)
            recs = make_pipe(trust, m, co=co).run(wf_test)
            fixed_fn = float(np.mean([r['final_call'] is not None and th.normalize_tool_call(r['final_call'])['function']
                                      == ex['ground_truth_tool_call']['function'] for r, ex in zip(recs, wf_test)]))
            rows.append({'w_semantic': w, 'min_score': ms, 'function_restored': fixed_fn,
                         'exact': float(np.mean([r['final_exact'] for r in recs]))})
    SENS[split_name] = {'test_wrong_function': rows,
                        'tune_all': TUNING_CURVES[split_name]['matcher'],
                        'selected': {k: TUNED[split_name][k] for k in ('w_semantic', 'min_score')}}
    print(f"\n[{split_name}] WRONG_FUNCTION test calls (n={len(wf_test)}): function name restored / exact")
    print(f"{'w_sem':>6} " + ''.join(f"{'min=' + str(ms):>16}" for ms in CFG['matcher']['min_score_grid']))
    for w in w_grid:
        cells = [r for r in rows if r['w_semantic'] == w]
        print(f"{w:>6} " + ''.join(f"{c['function_restored']:>8.3f}/{c['exact']:.3f}" for c in cells))
    print(f"selected on tune: {SENS[split_name]['selected']}")
RESULTS['matcher_sensitivity'] = SENS

## F. Type checking: strict vs skip-list vs adaptive, and the threshold sweep (#11)

* **Why conflicts occur** — published schemas and published example values disagree (e.g. ToolBench
  declares NUMBER but stores `"2"`); API-Bank declares `str` loosely; documentation is hand-written.
* **How a conflict is detected** — a *clean* tune call whose value type is incompatible with the declaration.
* **When type validation is relaxed** — adaptive mode: when the estimated trust of the declaration
  (pair → parameter name → declared type → prior back-off) is below τ.
* **Effect** — reported below as FPR on clean calls vs WRONG_TYPE recall, for each strategy and each τ.

In [ ]:
TYPE = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    tune = subset(splits[split_name]['tune'])
    _, matcher, co = tuned_parts(split_name)
    rows = {}
    for mode in ['off', 'strict', 'skiplist']:
        rows[mode] = fit_trust(tune, mode)
    for tau in CFG['type_trust']['tau_grid']:
        rows[f'adaptive tau={tau}'] = fit_trust(tune, 'adaptive', tau)
    TYPE[split_name] = {}
    th.print_header(f"TYPE-CHECK STRATEGIES — {split_name} test split")
    print(f"{'strategy':<22}{'FPR(clean)':>11}{'WRONG_TYPE recall':>19}{'detF1':>7}{'exact':>7}{'clean FP from WRONG_TYPE':>26}")
    for name, tt in rows.items():
        recs = make_pipe(tt, matcher, co=co).run(test)
        s = summarize(recs)
        wt = [r for r in recs if r['true_types'] == ['WRONG_TYPE']]
        wt_recall = float(np.mean(['WRONG_TYPE' in r['pred_types'] for r in wt])) if wt else None
        clean = [r for r in recs if not r['is_hallucinated']]
        fp_type = float(np.mean([any(v['type'] == 'WRONG_TYPE' for v in r['violations']) for r in clean]))
        TYPE[split_name][name] = {'summary': s, 'wrong_type_recall': wt_recall, 'clean_fp_due_to_type': fp_type}
        print(f"{name:<22}{s['fpr']:>11.3f}{wt_recall:>19.3f}{s['det_f1']:>7.3f}{s['exact_all']:>7.3f}{fp_type:>26.3f}")
    tt = fit_trust(tune, 'adaptive', TUNED[split_name]['tau'])
    levels = Counter()
    for ex in test:
        n = th.normalize_tool_call(ex['generated_tool_call'])
        fn = REG.resolve(n['function']) if n else None
        if not fn:
            continue
        for p in n['parameters']:
            spec = REG.params(fn).get(p)
            if isinstance(spec, dict) and spec.get('type'):
                levels[tt.level(fn, p, spec['type'])] += 1
    TYPE[split_name]['trust_level_used_on_test'] = dict(levels)
    print(f"trust estimate level used for test declarations: {dict(levels)}  "
          f"(function-disjoint test functions cannot use 'pair')")
RESULTS['type_checking'] = TYPE

## G. Schema-quality robustness (#10)

A fraction of all (function, parameter) declarations is corrupted (type flip, required flip,
declaration dropped, parameter renamed — one random kind each), the type trust is re-fitted on the
corrupted registry, and the frozen pipeline is re-run on the test part. 0 % = the registry as
published (it already contains natural inconsistencies). Three seeds per rate.

In [ ]:
NOISE = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    tune = subset(splits[split_name]['tune'])
    t = TUNED[split_name]
    _, matcher, co = tuned_parts(split_name)
    rows = []
    for rate in CFG['noise']['rates']:
        for seed in (CFG['noise']['seeds'] if rate > 0 else [0]):
            raw_c, log = th.corrupt_registry(raw_registry, rate, seed, CFG['noise']['kinds'])
            reg_c = th.Registry(raw_c, CFG['validator']['unknown_required_policy'])
            tt = fit_trust(tune, 'adaptive', t['tau'], registry=reg_c)
            m = matcher.configured(reg=reg_c)
            recs = make_pipe(tt, m, co=co, registry=reg_c).run(test)
            rows.append({'rate': rate, 'seed': seed, 'n_corrupted': log['n_corrupted'], **summarize(recs)})
    agg = {}
    for rate in CFG['noise']['rates']:
        rs = [r for r in rows if r['rate'] == rate]
        agg[rate] = {k: (float(np.mean([r[k] for r in rs])), float(np.std([r[k] for r in rs])))
                     for k in ('det_f1', 'fpr', 'fnr', 'class_macro_f1', 'exact_all', 'exact_hallucinated')}
    NOISE[split_name] = {'runs': rows, 'mean_std': {str(k): v for k, v in agg.items()}}
    th.print_header(f"SCHEMA NOISE — {split_name} test split (mean ± sd over seeds)")
    print(f"{'rate':>6}{'det F1':>16}{'FPR':>16}{'FNR':>16}{'macro-F1':>16}{'exact':>16}")
    for rate, a in agg.items():
        print(f"{rate:>6.2f}" + ''.join(f"{a[k][0]:>10.3f}±{a[k][1]:.3f}"
                                        for k in ('det_f1', 'fpr', 'fnr', 'class_macro_f1', 'exact_all')))
RESULTS['schema_noise'] = NOISE

## H. Multi-error calls (#2)

All multi-error calls use functions that are on the test side of the function-disjoint split, so
the function-disjoint tuned configuration is used. Two error-type predictions are reported:
* **first pass** — violation types found by the validator on the raw call (a WRONG_FUNCTION hides
  parameter errors because the parameters cannot be checked against an unknown function);
* **correction trace** — every type the corrector fixed plus any that remain.

In [ ]:
trust, matcher, co_tuned = tuned_parts('function_disjoint')
ACTION_TYPE = (('function', 'WRONG_FUNCTION'), ('removed', 'EXTRA_PARAMETER'), ('inserted', 'MISSING_PARAMETER'),
               ('cast', 'WRONG_TYPE'), ('restructured', 'WRONG_FORMAT'), ('snapped', 'WRONG_VALUE'),
               ('replaced', 'WRONG_VALUE'))


def multi_table(exs, co):
    recs = make_pipe(trust, matcher, co=co).run(exs)
    true_sets = [set(r['true_types']) for r in recs]
    first = [set(r['pred_types']) for r in recs]
    trace = []
    for r in recs:
        acted = {t for a in r['actions'] for prefix, t in ACTION_TYPE if a.startswith(prefix)}
        trace.append(acted | set(r['pred_types']))
    s = summarize(recs)
    return {'n': len(recs), 'detected': s['det_recall'], 'exact': s['exact_all'],
            'schema_valid': s['schema_valid_hallucinated'],
            'first_pass': th.multilabel_metrics(true_sets, first),
            'correction_trace': th.multilabel_metrics(true_sets, trace),
            'mean_iterations': float(np.mean([r['iterations'] for r in recs]))}


MAX_PASSES = max(CFG['corrector']['max_iterations_grid'])
MULTI = {'tuned_max_iterations': co_tuned['max_iterations'], 'max_passes': MAX_PASSES,
         'by_k': {}, 'by_k_max_passes': {}, 'by_passes': {}}
for k in CFG['benchmark']['multi_error_k']:
    exs = [e for e in multi if len(e['error_types']) == k]
    MULTI['by_k'][k] = multi_table(exs, co_tuned)
    MULTI['by_k_max_passes'][k] = multi_table(exs, {**co_tuned, 'max_iterations': MAX_PASSES})
    curve = []
    for it in CFG['corrector']['max_iterations_grid']:
        rr = make_pipe(trust, matcher, co={**co_tuned, 'max_iterations': it}).run(exs)
        curve.append({'max_iterations': it, 'exact': float(np.mean([r['final_exact'] for r in rr])),
                      'schema_valid': float(np.mean([r['final_schema_valid'] for r in rr]))})
    MULTI['by_passes'][k] = curve

th.print_header("MULTI-ERROR CALLS (function-disjoint tuned configuration)")
for table_key, title in (('by_k', f"tuned passes = {co_tuned['max_iterations']}"),
                         ('by_k_max_passes', f"max passes = {MAX_PASSES}")):
    print(f"--- {title} ---")
    print(f"{'k':>3}{'n':>5}{'detected':>10}{'schemaValid':>12}{'exact':>7}{'set-acc 1st':>13}{'set-acc trace':>15}"
          f"{'macroF1 1st':>13}{'macroF1 trace':>15}")
    for k, d in MULTI[table_key].items():
        print(f"{k:>3}{d['n']:>5}{d['detected']:>10.3f}{d['schema_valid']:>12.3f}{d['exact']:>7.3f}"
              f"{d['first_pass']['exact_set_accuracy']:>13.3f}{d['correction_trace']['exact_set_accuracy']:>15.3f}"
              f"{d['first_pass']['macro_f1']:>13.3f}{d['correction_trace']['macro_f1']:>15.3f}")
print("\nexact match by maximum number of correction passes:")
print(f"{'k':>3}" + ''.join(f"{'p=' + str(c['max_iterations']):>9}" for c in MULTI['by_passes'][1]))
for k, curve in MULTI['by_passes'].items():
    print(f"{k:>3}" + ''.join(f"{c['exact']:>9.3f}" for c in curve))
RESULTS['multi_error'] = MULTI

## I. WRONG_VALUE: schema only vs schema + user query (#6)

A schema can only reject values outside an enum or range; "Delhi" is a valid city even when the user
asked for Mumbai. Setting 2 adds a **query-grounding** check (a value that does not occur in the user
request is flagged) and replaces an ungrounded value with the best-matching span from the request.
Setting 3 (+ retrieval) is in Code_4. Results are split by whether the *reference* value itself
occurs in the query — ToolBench reference values are example values from the API documentation and
often do not, which bounds what any query-based method can do on this benchmark.

In [ ]:
WV = {}
for split_name in ['random', 'function_disjoint']:
    test = subset(splits[split_name]['test'])
    trust, matcher, co = tuned_parts(split_name)
    settings = {'1 schema only': {}, '2 schema + query': {'vo': {'check_grounding': True},
                                                          'co': {'fix_value_from_query': True}}}
    WV[split_name] = {}
    print(f"\n[{split_name}]  {'setting':<20}{'WV detected':>12}{'WV exact':>10}{'clean FPR':>11}{'all exact':>11}"
          f"{'WV exact | ref grounded':>25}{'| not grounded':>16}")
    for name, sp in settings.items():
        recs = make_pipe(trust, matcher, vo=sp.get('vo'), co={**co, **sp.get('co', {})}).run(test)
        wv = [(r, ex) for r, ex in zip(recs, test) if ex['error_types'] == ['WRONG_VALUE']]
        clean = [r for r in recs if not r['is_hallucinated']]
        g = [r for r, ex in wv if ex['injection'][0].get('gt_value_grounded') is True]
        ng = [r for r, ex in wv if ex['injection'][0].get('gt_value_grounded') is not True]
        d = {'wv_n': len(wv), 'wv_detected': float(np.mean([r['flagged'] for r, _ in wv])),
             'wv_exact': float(np.mean([r['final_exact'] for r, _ in wv])),
             'clean_fpr': float(np.mean([r['flagged'] for r in clean])),
             'all_exact': float(np.mean([r['final_exact'] for r in recs])),
             'wv_exact_ref_grounded': float(np.mean([r['final_exact'] for r in g])) if g else None,
             'wv_exact_ref_not_grounded': float(np.mean([r['final_exact'] for r in ng])) if ng else None,
             'n_ref_grounded': len(g), 'n_ref_not_grounded': len(ng)}
        WV[split_name][name] = d
        WV[split_name].setdefault('retrieval_targets', {})[name] = [
            {'id': r['id'], 'flagged_params': [f['param'] for f in r['flagged_for_retrieval']],
             'final_call': r['final_call'], 'final_exact_before': r['final_exact']}
            for r in recs if r['flagged_for_retrieval'] and r['final_call'] is not None]
        WV[split_name].setdefault('exact_by_id', {})[name] = {r['id']: bool(r['final_exact']) for r in recs}
        fmtv = lambda v: f"{v:.3f}" if v is not None else 'n/a'
        print(f"   {name:<20}{d['wv_detected']:>12.3f}{d['wv_exact']:>10.3f}{d['clean_fpr']:>11.3f}{d['all_exact']:>11.3f}"
              f"{fmtv(d['wv_exact_ref_grounded']):>25}{fmtv(d['wv_exact_ref_not_grounded']):>16}"
              f"   (n grounded={len(g)}, not={len(ng)})")
RESULTS['wrong_value'] = {k: {kk: vv for kk, vv in v.items() if kk not in ('retrieval_targets', 'exact_by_id')}
                          for k, v in WV.items()}

## J. Failure categories (random test split, frozen configuration)

In [ ]:
FAIL = {}
for split_name in ['random', 'function_disjoint']:
    cats = Counter(th.failure_category(r) for r in MAIN_RECORDS[split_name])
    by_type = defaultdict(Counter)
    for r in MAIN_RECORDS[split_name]:
        lab = r['true_types'][0] if r['true_types'] else 'NONE'
        by_type[lab][th.failure_category(r)] += 1
    FAIL[split_name] = {'overall': dict(cats), 'by_true_type': {k: dict(v) for k, v in by_type.items()}}
    n = len(MAIN_RECORDS[split_name])
    print(f"\n[{split_name}]")
    for c, v in sorted(cats.items(), key=lambda x: -x[1]):
        print(f"   {c:<32}{v:>6}  ({v / n:.1%})")
RESULTS['failures'] = FAIL

## K. Runtime per component (#15)

Measured on the main test run above (single process, this runtime's CPU). The matcher is timed
separately on every WRONG_FUNCTION test call.

In [ ]:
RUNTIME = {'hardware': th.hardware_info()}
for split_name in ['random', 'function_disjoint']:
    RUNTIME[split_name] = MAIN[split_name]['latency']
_, matcher, _ = tuned_parts('random')
matcher = matcher.fresh()
mt = []
for ex in subset(splits['random']['test']):
    if 'WRONG_FUNCTION' in ex['error_types']:
        n = th.normalize_tool_call(ex['generated_tool_call'])
        t1 = time.perf_counter()
        matcher.match(n['function'])
        mt.append(time.perf_counter() - t1)
RUNTIME['matcher_only'] = th.latency_summary({'match': mt})['match'] if mt else None
print(json.dumps(th.strip_ci(RUNTIME), indent=1))
RESULTS['runtime'] = RUNTIME

## Save

In [ ]:
def slim(r):
    return {k: r[k] for k in ('id', 'function_key', 'is_hallucinated', 'true_types', 'flagged', 'pred_types',
                              'pred_primary', 'final_call', 'final_schema_valid', 'final_exact',
                              'iterations', 'actions', 'uncertain_params', 'flagged_for_retrieval')}

records_out = {s: [slim(r) for r in MAIN_RECORDS[s]] for s in MAIN_RECORDS}
records_out['retrieval_targets'] = {s: WV[s]['retrieval_targets'] for s in WV}
records_out['wrong_value_exact_by_id'] = {s: WV[s]['exact_by_id'] for s in WV}
th.save_json(RESULTS, P.run_file('core_results.json'), CFG, NOTEBOOK)
th.save_json(records_out, P.run_file('core_records.json'), CFG, NOTEBOOK)
print('saved ->', P.run_file('core_results.json'))
print('saved ->', P.run_file('core_records.json'))
print('Code_3 complete.')